## 6. Testgebiet auf Karte visualisieren

**Was sehen wir hier?**
- Diese Karte dient in **Sprint 1** als rein geografischer Rahmen (Bounding Box).
- Es werden noch **keine Copernicus-Rasterlayer** angezeigt, da wir diese erst in **Sprint 2** herunterladen.
- **Grün:** Nationalpark Donau-Auen (eigentliches Untersuchungsgebiet).
- **Orange gestrichelt:** Download-Bounding-Box mit 5 km Puffer für saubere Ränder.
- **Basemaps:** OpenStreetMap & basemap.at (Österreichisches Orthofoto/Luftbild) — komplett ohne API-Key.


In [1]:
# Autoreload aktivieren, damit Änderungen im src/ Code sofort übernommen werden
%load_ext autoreload
%autoreload 2

# Standard-Imports & Logging konfigurieren
import logging
import sys
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')  # Für saubere Notebook-Ausgabe

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('sprint_01')

# Projekt-Root zum Python-Pfad hinzufügen
project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

print(f'✅ Project root: {project_root}')
print(f'✅ Python: {sys.version.split()[0]}')

✅ Project root: C:\Users\danie\Projekte\private\biodiv-horizon
✅ Python: 3.12.14


In [2]:
# Konfiguration laden
from biodiv_horizon.config import (
    CDSE_STAC_URL,
    TEST_AREA_BBOX,
    TEST_AREA_NAME,
    DOWNLOAD_BBOX,
    CLMS_LAYERS,
    LEITARTEN,
    COPERNICUS_DIR,
)

print(f'📍 Testgebiet: {TEST_AREA_NAME}')
print(f'📦 BBox (WGS84): {TEST_AREA_BBOX}')
print(f'📦 Download-BBox (mit Puffer): {DOWNLOAD_BBOX}')
print(f'🌿 Leitarten: {[v["common_name_de"] for v in LEITARTEN.values()]}')
print(f'🗂️  CLMS Layer: {list(CLMS_LAYERS.keys())}')
print(f'📁 Copernicus Daten-Dir: {COPERNICUS_DIR}')

📍 Testgebiet: Nationalpark Donau-Auen
📦 BBox (WGS84): [16.5, 48.08, 16.95, 48.22]
📦 Download-BBox (mit Puffer): [16.45, 48.03, 17.0, 48.269999999999996]
🌿 Leitarten: ['Feldhase', 'Fasan', 'Admiral', 'Neuntöter']
🗂️  CLMS Layer: ['tree_cover', 'grassland', 'forest_type', 'imperviousness', 'water_wetness']
📁 Copernicus Daten-Dir: C:\Users\danie\Projekte\private\biodiv-horizon\data\raw\copernicus


## 1. CDSE STAC-Katalog verbinden

Der Copernicus Data Space Ecosystem (CDSE) stellt einen STAC-kompatiblen Katalog bereit:
- **URL:** https://catalogue.dataspace.copernicus.eu/stac
- **Keine Authentifizierung** für Metadaten-Abfragen nötig
- **Login** nur für den eigentlichen Download von Rohdaten (Sentinel-1/2 etc.)
- **CLMS HRL** (High Resolution Layers) sind als voraggregierte Produkte verfügbar


In [3]:
from pystac_client import Client

print(f'Verbinde mit CDSE STAC: {CDSE_STAC_URL}')
catalog = Client.open(CDSE_STAC_URL)

print(f'✅ Verbunden!')
print(f'   Titel:   {catalog.title}')
print(f'   Beschr.: {catalog.description[:100] if catalog.description else "N/A"}')

Verbinde mit CDSE STAC: https://catalogue.dataspace.copernicus.eu/stac
✅ Verbunden!
   Titel:   Copernicus Data Space Ecosystem (CDSE) asset-level STAC catalogue
   Beschr.: A comprehensive and searchable catalog of Earth observation and scientific datasets that is actively


## 2. Verfügbare Collections auflisten

Wir schauen uns an, welche Datensammlungen (Collections) der CDSE-Katalog anbietet.
Für uns relevant sind primär **CLMS**-Collections (Copernicus Land Monitoring Service).

In [4]:
import pandas as pd

print('Lade alle verfügbaren Collections...')
collections = list(catalog.get_collections())
print(f'\n✅ {len(collections)} Collections gefunden:')

# Als DataFrame darstellen
coll_data = []
for c in collections:
    coll_data.append({
        'ID': c.id,
        'Titel': c.title or 'N/A',
        'Beschreibung': (c.description or '')[:80],
    })

df_collections = pd.DataFrame(coll_data)

# CLMS-Collections hervorheben
clms_mask = df_collections['ID'].str.upper().str.contains('CLMS|LAND|HRL')
print(f'\n🌿 CLMS/Land-relevante Collections ({clms_mask.sum()} Stück):')
print(df_collections[clms_mask].to_string(index=False))

print(f'\n📋 Alle Collections:')
print(df_collections.to_string(index=False))

15:27:18 [INFO] numexpr.utils: NumExpr defaulting to 16 threads.


Lade alle verfügbaren Collections...

✅ 422 Collections gefunden:

🌿 CLMS/Land-relevante Collections (224 Stück):
                                                                         ID                                                                                                          Titel                                                                     Beschreibung
                                           clms_ba_global_300m_daily_v3_cog                                                                CLMS Burnt Area (BA) Global 300m daily V3 (COG) Maps burn scars, surfaces which have been sufficiently affected by fire to displ
                                            clms_ba_global_300m_daily_v3_nc                                                             CLMS Burnt Area (BA) Global 300m daily V3 (NetCDF) Maps burn scars, surfaces which have been sufficiently affected by fire to displ
                                           clms_ba_global_300m_daily_v4_cog       

## 3. CLMS-Produkte für Testgebiet suchen

Wir suchen gezielt nach CLMS-Items für:
- **Bounding Box:** Nationalpark Donau-Auen + 5km Puffer
- **Zeitraum:** 2018–2023 (Referenzjahre für HRL-Produkte)
- **Produkte:** Imperviousness, Tree Cover Density, Grassland, Water & Wetness

In [7]:
from biodiv_horizon.ingestion.copernicus import (
    DEFAULT_CLMS_COLLECTIONS,
    search_clms_items,
    print_item_summary,
)

print(f'Verwendete Collections ({len(DEFAULT_CLMS_COLLECTIONS)}):')
for c in DEFAULT_CLMS_COLLECTIONS:
    print(f'  - {c}')

# Suche über verschiedene Zeiträume (HRL-Produkte erscheinen alle 3 Jahre bzw. jährlich)
search_configs = [
    ('HRL 2018', '2018-01-01/2019-12-31'),
    ('HRL 2021', '2021-01-01/2022-12-31'),
]

all_items = []
for label, dt_range in search_configs:
    print(f'\n🔍 Suche {label}: {dt_range}')
    items = search_clms_items(catalog, bbox=DOWNLOAD_BBOX, datetime_range=dt_range)
    print(f'   → {len(items)} Items gefunden')
    all_items.extend(items)

print(f'\n📦 Gesamt: {len(all_items)} Items')
print_item_summary(all_items)

Verwendete Collections (3):
  - clms_vlcc_tree-cover-density_europe_10m_yearly_v1
  - clms_vlcc_grassland_europe_10m_yearly_v1
  - clms_vlcc_forest-type_europe_10m_3yearly_v1

🔍 Suche HRL 2018: 2018-01-01/2019-12-31


15:27:52 [INFO] biodiv_horizon.ingestion.copernicus: Collection 'clms_vlcc_tree-cover-density_europe_10m_yearly_v1': 6 Items gefunden
15:27:52 [INFO] biodiv_horizon.ingestion.copernicus: Collection 'clms_vlcc_grassland_europe_10m_yearly_v1': 6 Items gefunden
15:27:53 [INFO] biodiv_horizon.ingestion.copernicus: Collection 'clms_vlcc_forest-type_europe_10m_3yearly_v1': 3 Items gefunden
15:27:53 [INFO] biodiv_horizon.ingestion.copernicus: Collection 'clms_vlcc_tree-cover-density_europe_10m_yearly_v1': 6 Items gefunden


   → 15 Items gefunden

🔍 Suche HRL 2021: 2021-01-01/2022-12-31


15:27:54 [INFO] biodiv_horizon.ingestion.copernicus: Collection 'clms_vlcc_grassland_europe_10m_yearly_v1': 6 Items gefunden
15:27:55 [INFO] biodiv_horizon.ingestion.copernicus: Collection 'clms_vlcc_forest-type_europe_10m_3yearly_v1': 3 Items gefunden


   → 15 Items gefunden

📦 Gesamt: 30 Items

ID                                                 Datum        Assets                        
-----------------------------------------------------------------------------------------------
CLMS_HRLVLCC_TCD_S2019_R10m_E48N28_03035_V01_R00   2019-01-01   data, product, metadata, thumbnail
CLMS_HRLVLCC_TCD_S2019_R10m_E48N27_03035_V01_R00   2019-01-01   data, product, metadata, thumbnail
CLMS_HRLVLCC_TCD_S2019_R10m_E47N28_03035_V01_R00   2019-01-01   data, product, metadata, thumbnail
CLMS_HRLVLCC_TCD_S2018_R10m_E48N28_03035_V01_R00   2018-01-01   data, product, metadata, thumbnail
CLMS_HRLVLCC_TCD_S2018_R10m_E48N27_03035_V01_R00   2018-01-01   data, product, metadata, thumbnail
CLMS_HRLVLCC_TCD_S2018_R10m_E47N28_03035_V01_R00   2018-01-01   data, product, metadata, thumbnail
CLMS_HRLVLCC_GRA_S2019_R10m_E48N28_03035_V01_R01   2019-01-01   data, product, metadata, thumbnail
CLMS_HRLVLCC_GRA_S2019_R10m_E48N27_03035_V01_R01   2019-01-01   data, pr

In [10]:
# Item-Details anzeigen: Assets (Download-Links)
if all_items:
    print('\n🔎 Detail-Ansicht der gefundenen Items:')
    for item in all_items[:5]:  # Erste 5 Items
        print(f'\n--- {item.id} ---')
        print(f'  Datum:      {item.datetime}')
        print(f'  BBox:       {item.bbox}')
        print(f'  Properties: {dict(list(item.properties.items())[:5])}')
        print(f'  Assets:')
        for asset_key, asset in item.assets.items():
            size = asset.extra_fields.get('file:size', 'N/A')
            print(f'    [{asset_key}] {asset.href[:80]} (Größe: {size})')
else:
    print('\n⚠️  Keine STAC-Items gefunden. Sieh Abschnitt 4 für alternative Datenquellen.')


🔎 Detail-Ansicht der gefundenen Items:

--- CLMS_HRLVLCC_TCD_S2019_R10m_E48N28_03035_V01_R00 ---
  Datum:      2019-01-01 00:00:00+00:00
  BBox:       [16.4404963, 48.0389557, 17.9173921, 49.0210512]
  Properties: {'gsd': 10, 'created': '2025-12-17T23:43:50.237425Z', 'expires': '2262-01-01T00:00:00.000000Z', 'updated': '2025-12-22T11:14:56.618633Z', '_private': {'cellID': '100kmE48N28', 'visible': True, 'component': 'landcover_landuse', 'gridLabel': '10m', 'fileFormat': 'cog', 'nominalDate': '2019-01-01T00:00:00.000000Z', 'datasetAlias': 'clms_vlcc_tree-cover-density', 'product_size': 25095244, 'product_uuid': '155c5b43-dfb1-4328-b9ce-cda4ce2de003', 'areaOfInterest': 'europe', 'collectionName': 'CLMS', 'datasetVersion': 1, 'productVersion': 'v01_r00', 'platformAcronym': 'unspecified', 'datasetShortName': 'tcd', 'missionShortName': 'unspecified', 'processingCenter': 'unspecified', 'datasetIdentifier': 'clms_vlcc_tree-cover-density_europe_10m_yearly_v1', 'metricGridSpacing': 10, 'platfo

## 4. Alternative Datenquellen (falls STAC keine Ergebnisse liefert)

Der CDSE-STAC-Katalog ist primär für **Sentinel-Rohdaten** optimiert.
Die voraggregierten **CLMS High Resolution Layer (HRL)** sind auch über andere Kanäle verfügbar:

| Quelle | URL | Direkt-Download? |
|:---|:---|:---|
| **EEA DiscoMap (ArcGIS REST)** | https://image.discomap.eea.europa.eu/arcgis/rest/services/GioLandPublic | ✅ ImageServer exportImage |
| **Copernicus CLMS Portal** | https://land.copernicus.eu/en/products | ✅ Manuell |
| **OpenEO (CDSE)** | https://openeo.dataspace.copernicus.eu | ✅ API |
| **WEkEO** | https://www.wekeo.eu/ | ✅ HDA-API |


In [11]:
# EEA DiscoMap ImageServer (ArcGIS REST API) testen
# Hinweis: Die OGC-WCS-Extension (WCSServer) ist auf dem EEA-Server deaktiviert (HTTP 400).
# EEA stellt die Daten stattdessen über die native ImageServer REST API bereit:
# - Metadaten: ?f=json
# - GeoTIFF-Export: /exportImage?bbox=...&format=tiff&f=image
import requests

eea_services = {
    'Imperviousness 2018': 'https://image.discomap.eea.europa.eu/arcgis/rest/services/GioLandPublic/HRL_ImperviousnessDensity_2018/ImageServer',
    'Tree Cover Density 2018': 'https://image.discomap.eea.europa.eu/arcgis/rest/services/GioLandPublic/HRL_TreeCoverDensity_2018/ImageServer',
    'Grassland 2018': 'https://image.discomap.eea.europa.eu/arcgis/rest/services/GioLandPublic/HRL_Grassland_2018/ImageServer',
    'Water & Wetness 2018': 'https://image.discomap.eea.europa.eu/arcgis/rest/services/GioLandPublic/HRL_WaterWetness_2018/ImageServer',
}

print('🌐 EEA DiscoMap Erreichbarkeits-Check (REST ImageServer):')
for name, url in eea_services.items():
    try:
        resp = requests.get(url, params={'f': 'json'}, timeout=15)
        if resp.status_code == 200:
            svc_name = resp.json().get('name', '')
            print(f'  ✅ {name:25s} -> {svc_name}')
        else:
            print(f'  ⚠️  HTTP {resp.status_code} {name}')
    except requests.exceptions.Timeout:
        print(f'  ⏰ Timeout: {name}')
    except requests.exceptions.ConnectionError:
        print(f'  ❌ Keine Verbindung: {name}')


🌐 EEA DiscoMap Erreichbarkeits-Check (REST ImageServer):
  ✅ Imperviousness 2018       -> GioLandPublic/HRL_ImperviousnessDensity_2018
  ✅ Tree Cover Density 2018   -> GioLandPublic/HRL_TreeCoverDensity_2018
  ✅ Grassland 2018            -> GioLandPublic/HRL_Grassland_2018
  ✅ Water & Wetness 2018      -> GioLandPublic/HRL_WaterWetness_2018


## 5. OpenEO als Alternative (EU-Standard-API für Rasterdaten)

**OpenEO** ist der EU-Standard für Cloud-basierte Erdbeobachtungsdaten.
CDSE stellt einen kostenlosen OpenEO-Endpunkt bereit.
Für **voraggregierte CLMS-Layer** ist OpenEO oft komfortabler als STAC.

In [12]:
# OpenEO CDSE verbinden und verfügbare Collections anzeigen
try:
    import openeo
    
    print('Verbinde mit OpenEO CDSE...')
    conn = openeo.connect('https://openeo.dataspace.copernicus.eu')
    
    collections = conn.list_collections()
    print(f'✅ {len(collections)} Collections via OpenEO:')
    
    # CLMS-relevante Collections
    clms_cols = [c for c in collections if 'CLMS' in c.get('id', '').upper() or 'LAND' in c.get('id', '').upper()]
    print(f'\n🌿 CLMS/Land Collections ({len(clms_cols)}):')
    for c in clms_cols:
        print(f'  {c["id"]}: {c.get("title", "N/A")}')

except ImportError:
    print('ℹ️  openeo nicht installiert. Installieren mit: uv add openeo')
    print('   Für Sprint 1 nicht zwingend erforderlich.')
except Exception as e:
    print(f'⚠️  OpenEO-Verbindung fehlgeschlagen: {e}')

ℹ️  openeo nicht installiert. Installieren mit: uv add openeo
   Für Sprint 1 nicht zwingend erforderlich.


## 6. Testgebiet auf Karte visualisieren

**Was sehen wir hier?**
- Diese Karte dient in **Sprint 1** als rein geografischer Rahmen (Bounding Box).
- Es werden noch **keine Copernicus-Rasterlayer** angezeigt, da wir diese erst in **Sprint 2** herunterladen.
- **Grün:** Nationalpark Donau-Auen (eigentliches Untersuchungsgebiet).
- **Orange gestrichelt:** Download-Bounding-Box mit 5 km Puffer für saubere Ränder.
- **Basemaps:** OpenStreetMap & basemap.at (Österreichisches Orthofoto/Luftbild) — komplett ohne API-Key.


In [ ]:
import folium
from shapely.geometry import box

# Karte zentriert auf Testgebiet
center_lat = (TEST_AREA_BBOX[1] + TEST_AREA_BBOX[3]) / 2
center_lon = (TEST_AREA_BBOX[0] + TEST_AREA_BBOX[2]) / 2

# OpenStreetMap als Standard (100% Open Source & kostenlos, kein API-Key)
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='OpenStreetMap',
)

# Amtliche österreichische Basemap (basemap.at Standard & Orthofoto)
folium.TileLayer(
    tiles='https://mapsneu.wien.gv.at/basemap/geolandbasemap/normal/google3857/{z}/{y}/{x}.png',
    attr='Datenquelle: basemap.at',
    name='basemap.at (Standard)',
    overlay=False,
    control=True,
).add_to(m)

folium.TileLayer(
    tiles='https://mapsneu.wien.gv.at/basemap/bmaporthofoto30cm/normal/google3857/{z}/{y}/{x}.jpeg',
    attr='Datenquelle: basemap.at Orthofoto',
    name='basemap.at Orthofoto (Luftbild)',
    overlay=False,
    control=True,
).add_to(m)

# Testgebiet-BBox (Grün)
folium.Rectangle(
    bounds=[[TEST_AREA_BBOX[1], TEST_AREA_BBOX[0]], [TEST_AREA_BBOX[3], TEST_AREA_BBOX[2]]],
    color='#22c55e',
    weight=3,
    fill=True,
    fill_opacity=0.15,
    tooltip=f'Testgebiet: {TEST_AREA_NAME}',
    popup=f'<b>{TEST_AREA_NAME}</b><br>BBox: {TEST_AREA_BBOX}<br>Fläche: ~93 km²',
).add_to(m)

# Download-BBox (mit 5km Puffer - Orange gestrichelt)
folium.Rectangle(
    bounds=[[DOWNLOAD_BBOX[1], DOWNLOAD_BBOX[0]], [DOWNLOAD_BBOX[3], DOWNLOAD_BBOX[2]]],
    color='#f59e0b',
    weight=2,
    dash_array='10 5',
    fill=False,
    tooltip='Download-BBox (5km Puffer)',
).add_to(m)

# Umschalter für Basemaps oben rechts
folium.LayerControl(position='topright').add_to(m)

# Legende
legend_html = '''
<div style="position: fixed; bottom: 30px; right: 30px; 
            background: rgba(0,0,0,0.8); color: white; 
            padding: 12px; border-radius: 8px; font-size: 13px; z-index: 9999;">
    <b>BioDiv-Horizon Testgebiet</b><br>
    <span style="color:#22c55e">■</span> Nationalpark Donau-Auen<br>
    <span style="color:#f59e0b">- -</span> Download-BBox (+5km Puffer)
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f'🗺️ Karte für: {TEST_AREA_NAME}')
print(f'   Center: [{center_lat:.4f}, {center_lon:.4f}]')
m


## 7. Zusammenfassung & Entscheidung

| Datenzugang | Status | Nächster Schritt |
|:---|:---|:---|
| **CDSE STAC (Metadaten)** | ✅ Verbunden | Collections analysiert |
| **CLMS via CDSE STAC** | ✅ Verfügbar | Tree Cover, Grassland, Forest Type |
| **EEA DiscoMap (REST)** | ✅ Verfügbar | Imperviousness, Water & Wetness via exportImage |
| **OpenEO CDSE** | Optional | Für komplexe Cloud-Zeitreihen |


In [ ]:
# Zusammenfassung: was haben wir und was brauchen wir für Sprint 2?
print('=' * 60)
print('SPRINT 1 — ZUSAMMENFASSUNG')
print('=' * 60)
print()
print(f'📍 Testgebiet: {TEST_AREA_NAME}')
print(f'📦 BBox: {TEST_AREA_BBOX}')
print()
print('🌿 Benötigte CLMS-Layer für Sprint 2:')
for key, info in CLMS_LAYERS.items():
    src = info.get('source', 'n/a').upper()
    print(f'   • {key:15s} [{src}]: {info["description"]}')
print()
print('🐾 Leitarten für Sprint 3:')
for species, info in LEITARTEN.items():
    ffh_flag = '(FFH Anhang I)' if info['ffh'] else ''
    print(f'   {info["emoji"]} {info["common_name_de"]} ({species}) — GBIF: {info["taxon_key"]} {ffh_flag}')
print()
print('➡️  Nächster Schritt: notebooks/02_download_clms_layers.ipynb')
print('   → CLMS-Raster herunterladen und als COG speichern')
